##### Setup
- Explore which factors (columns) are associated with `cancel_yn` (churn), using the full cleaned dataset (no train/test split needed for this exploratory step)

In [2]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', None)

CLEAN = "read_parquet('df_clean_part*.parquet')"

##### Baseline Churn Rate
- Reference point for comparing segment-level churn rates below

In [ ]:
baseline = duckdb.sql(f"""
    SELECT round(count(*) FILTER (WHERE cancel_yn = '해지') * 100.0 / count(*), 4) AS baseline_churn_rate_pct
    FROM {CLEAN}
""").df()
baseline

,baseline_churn_rate_pct
0,4.0343


##### Categorical Columns: Churn Rate by Category
- For every categorical column, compute the churn rate (%) within each category value
- `deviation_from_baseline` shows how far each segment's churn rate is from the overall average — sorted by absolute deviation, so the strongest factors surface first
- Filters out tiny groups (`n < 1000`) to avoid noisy small-sample rates

In [7]:
categorical_cols = [
    "SVC_USE_DAYS_GRP", "MEDIA_NM_GRP", "PROD_NM_GRP", "PROD_OLD_YN", "PROD_ONE_PLUS_YN",
    "AGMT_KIND_NM", "STB_RES_1M_YN", "SVOD_SCRB_CNT_GRP", "PAID_CHNL_CNT_GRP", "SCRB_PATH_NM_GRP",
    "AGMT_END_SEG", "BUNDLE_YN", "DIGITAL_GIGA_YN", "DIGITAL_ALOG_YN", "CH_LAST_DAYS_BF_GRP",
    "VOC_TOTAL_MONTH1_YN", "VOC_STOP_CANCEL_MONTH1_YN", "AGE_GRP10", "EMAIL_RECV_CLS_NM",
    "SMS_SEND_CLS_NM", "NFX_USE_YN", "YTB_USE_YN", "CH_FAV_RNK1",
]
# SQL에 들어가기 위해 여러 컬럼을 조회할 때 대괄호 없이 쉼표로 연결된 일반 텍스트
cols_sql = ", ".join(categorical_cols)
baseline_rate = baseline['baseline_churn_rate_pct'][0]

categorical_churn_scan = duckdb.sql(f"""
    WITH base AS (
        SELECT
            CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END AS is_cancelled,
            {cols_sql}
        FROM {CLEAN}
    ),
    unpvt AS (
        UNPIVOT base
        ON {cols_sql}
        INTO
            NAME column_name
            VALUE value
    )
    SELECT
        column_name,
        value,
        count(*) AS n,
        round(avg(is_cancelled) * 100, 4) AS churn_rate_pct,
        round(avg(is_cancelled) * 100 - {baseline_rate}, 4) AS deviation_from_baseline,
        round(avg(is_cancelled) * 100 / {baseline_rate}, 2) AS ratio_to_baseline
    FROM unpvt
    GROUP BY column_name, value
    HAVING count(*) >= 1000
    ORDER BY abs(round(avg(is_cancelled) * 100 - {baseline_rate}, 4)) DESC
""").df()

categorical_churn_scan.head(30)

,column_name,value,n,churn_rate_pct,deviation_from_baseline,ratio_to_baseline
0,AGMT_END_SEG,약정만료전 1개월,181220,17.6548,13.6205,4.38
1,AGMT_END_SEG,약정만료전 1~2개월,205110,14.8886,10.8543,3.69
2,VOC_STOP_CANCEL_MONTH1_YN,Y,464918,13.8674,9.8331,3.44
3,AGMT_END_SEG,약정만료전 2~3개월,224681,12.0250,7.9907,2.98
4,AGE_GRP10,20대,185727,11.7560,7.7217,2.91
5,AGMT_END_SEG,약정만료후 1개월~2개월,128414,9.4686,5.4343,2.35
6,AGMT_END_SEG,약정만료 1개월,328269,8.9935,4.9592,2.23
7,PROD_NM_GRP,세이버,1652,8.4746,4.4403,2.10
8,AGMT_END_SEG,약정만료후 2개월~3개월,118056,8.3232,4.2889,2.06
9,AGMT_END_SEG,약정만료전 3~6개월,795058,8.2893,4.2550,2.05


##### Finding

`AGMT_END_SEG` shows up 8 times in the top 30, so we should check for multicollinearity before treating these as 8 separate signals.

A disproportionate share of cancellations come from customers in their 20s — this age group could be a good target for retention marketing.

`VOC_STOP_CANCEL_MONTH1_YN` isn't very useful on its own, since it only flags customers who've already decided to cancel. It'd be more valuable to look at why some customers who call about cancelling end up *not* cancelling.

##### Numeric Columns: Average Value by cancel_yn
- Compare the average of each numeric column between the `해지`(cancelled) and `유지`(retained) groups
- A large gap between the two rows suggests that column is informative for churn prediction

In [ ]:
numeric_cols = [
    "TOTAL_USED_DAYS", "TV_SCRB", "ANALOG_SCRB", "DIGITAL_SCRB",
    "TOTAL_INTERNET_SCRB", "GIGA_INTERNET_SCRB", "TV_I_CNT",
    "CH_HH_AVG_MONTH1", "CH_25_RATIO_MONTH1", "CH_25_RATIO_MEAN_3MM",
    "KIDS_USE_PV_MONTH1", "INHOME_RATE",
]

parts = []
for c in numeric_cols:
    expr = "round(avg(" + c + "), 3) AS avg_" + c
    parts.append(expr)

avg_exprs = ", ".join(parts)


numeric_churn_scan = duckdb.sql(f"""
    SELECT cancel_yn, count(*) AS n, {avg_exprs}
    FROM {CLEAN}
    GROUP BY cancel_yn
""").df()

numeric_churn_scan.T

,0,1
cancel_yn,해지,유지
n,889705,21163859
avg_TOTAL_USED_DAYS,2300.376,2777.906
avg_TV_SCRB,1.716,1.782
avg_ANALOG_SCRB,0.016,0.021
avg_DIGITAL_SCRB,1.7,1.761
avg_TOTAL_INTERNET_SCRB,0.656,0.489
avg_GIGA_INTERNET_SCRB,0.249,0.129
avg_TV_I_CNT,2.371,2.271
avg_CH_HH_AVG_MONTH1,3.218,4.613


##### Findings


##### Do Customers Still Appear After Their First `해지` Month?
- For each customer's first `해지` month, check whether they have any later row (`p_mt` after that month)
- If almost nobody appears afterward, that confirms `해지` = "left the dataset," so each churning customer contributes only one `해지` row out of their whole history

In [8]:
after_cancel_check = duckdb.sql(f"""
    WITH first_cancel AS (
        SELECT sha2_hash, min(p_mt) AS first_cancel_pmt
        FROM {CLEAN}
        WHERE cancel_yn = '해지'
        GROUP BY sha2_hash
    ),
    joined AS (
        SELECT
            fc.sha2_hash,
            fc.first_cancel_pmt,
            count(*) FILTER (WHERE c.p_mt > fc.first_cancel_pmt) AS n_rows_after
        FROM first_cancel fc
        JOIN {CLEAN} c ON c.sha2_hash = fc.sha2_hash
        GROUP BY fc.sha2_hash, fc.first_cancel_pmt
    )
    SELECT
        count(*) AS n_customers_with_cancel,
        count(*) FILTER (WHERE n_rows_after = 0) AS n_no_rows_after,
        count(*) FILTER (WHERE n_rows_after > 0) AS n_has_rows_after,
        round(count(*) FILTER (WHERE n_rows_after = 0) * 100.0 / count(*), 2) AS pct_no_rows_after
    FROM joined
""").df()

after_cancel_check

,n_customers_with_cancel,n_no_rows_after,n_has_rows_after,pct_no_rows_after
0,150613,15900,134713,10.56


##### What Does cancel_yn Look Like in the Rows After First `해지`?
- For the 134,713 customers who have rows after their first `해지`, check whether those later rows are mostly `유지` (reverted / reactivated) or mostly `해지` (status persists)

In [ ]:
rows_after_status = duckdb.sql(f"""
    WITH first_cancel AS (
        SELECT sha2_hash, min(p_mt) AS first_cancel_pmt
        FROM {CLEAN}
        WHERE cancel_yn = '해지'
        GROUP BY sha2_hash
    ),
    rows_after AS (
        SELECT c.cancel_yn
        FROM {CLEAN} c
        JOIN first_cancel fc ON c.sha2_hash = fc.sha2_hash
        WHERE c.p_mt > fc.first_cancel_pmt
    )
    SELECT
        cancel_yn,
        count(*) AS n,
        round(count(*) * 100.0 / sum(count(*)) OVER (), 2) AS pct
    FROM rows_after
    GROUP BY cancel_yn
""").df()

rows_after_status

,cancel_yn,n,pct
0,해지,739092,100.0


##### Do the Other Columns Also Freeze After First `해지`?
- We already know `cancel_yn` stays `해지` in every row after the first cancellation. Do the other columns (`TV_I_CNT`, `CH_HH_AVG_MONTH1`, etc.) also stay frozen at the same value, or do they keep updating month to month?
- For customers with 2+ rows after their first `해지`, check how many distinct values each column takes across those rows — `n_distinct=1` means frozen

In [10]:
other_cols = [
    "SVC_USE_DAYS_GRP", "MEDIA_NM_GRP", "PROD_NM_GRP", "PROD_OLD_YN", "PROD_ONE_PLUS_YN",
    "AGMT_KIND_NM", "STB_RES_1M_YN", "SVOD_SCRB_CNT_GRP", "PAID_CHNL_CNT_GRP", "SCRB_PATH_NM_GRP",
    "INHOME_RATE", "AGMT_END_SEG", "AGMT_END_YMD", "TOTAL_USED_DAYS", "TV_SCRB", "ANALOG_SCRB",
    "DIGITAL_SCRB", "TOTAL_INTERNET_SCRB", "GIGA_INTERNET_SCRB", "BUNDLE_YN", "DIGITAL_GIGA_YN",
    "DIGITAL_ALOG_YN", "TV_I_CNT", "CH_LAST_DAYS_BF_GRP", "VOC_TOTAL_MONTH1_YN",
    "VOC_STOP_CANCEL_MONTH1_YN", "AGE_GRP10", "EMAIL_RECV_CLS_NM", "SMS_SEND_CLS_NM",
    "CH_HH_AVG_MONTH1", "CH_25_RATIO_MONTH1", "CH_25_RATIO_MEAN_3MM", "CH_FAV_RNK1",
    "KIDS_USE_PV_MONTH1", "NFX_USE_YN", "YTB_USE_YN",
]
# INHOME_RATE/CH_25_RATIO_MONTH1/etc. are DOUBLE while the rest are VARCHAR in the
# cleaned data — UNPIVOT needs matching types, so cast everything to VARCHAR first
other_cols_sql = ", ".join(other_cols)
cast_cols_sql = ", ".join(f"CAST({c} AS VARCHAR) AS {c}" for c in other_cols)

frozen_check = duckdb.sql(f"""
    WITH raw AS (
        SELECT sha2_hash, p_mt, cancel_yn, {cast_cols_sql}
        FROM {CLEAN}
    ),
    first_cancel AS (
        SELECT sha2_hash, min(p_mt) AS first_cancel_pmt
        FROM raw
        WHERE cancel_yn = '해지'
        GROUP BY sha2_hash
    ),
    rows_after AS (
        SELECT r.*
        FROM raw r
        JOIN first_cancel fc ON r.sha2_hash = fc.sha2_hash
        WHERE r.p_mt > fc.first_cancel_pmt
    ),
    customers_with_2plus AS (
        SELECT sha2_hash
        FROM rows_after
        GROUP BY sha2_hash
        HAVING count(*) >= 2
    ),
    unpvt AS (
        UNPIVOT (
            SELECT sha2_hash, {other_cols_sql}
            FROM rows_after
            WHERE sha2_hash IN (SELECT sha2_hash FROM customers_with_2plus)
        )
        ON {other_cols_sql}
        INTO
            NAME column_name
            VALUE value
    ),
    per_customer_col AS (
        SELECT sha2_hash, column_name, count(DISTINCT value) AS n_distinct
        FROM unpvt
        GROUP BY sha2_hash, column_name
    )
    SELECT
        column_name,
        count(*) AS n_customers,
        round(avg(CASE WHEN n_distinct = 1 THEN 100.0 ELSE 0 END), 2) AS pct_frozen
    FROM per_customer_col
    GROUP BY column_name
    ORDER BY pct_frozen
""").df()

frozen_check

,column_name,n_customers,pct_frozen
0,TOTAL_USED_DAYS,120987,0.00
1,CH_HH_AVG_MONTH1,120987,23.96
2,CH_25_RATIO_MONTH1,120987,29.74
3,CH_25_RATIO_MEAN_3MM,120987,29.74
4,VOC_TOTAL_MONTH1_YN,120987,33.62
5,INHOME_RATE,95899,41.10
6,CH_FAV_RNK1,120987,49.16
7,AGMT_END_SEG,120987,59.83
8,CH_LAST_DAYS_BF_GRP,120987,60.59
9,VOC_STOP_CANCEL_MONTH1_YN,120987,64.33
